In [1]:
import pandas as pd

df = pd.read_parquet("../data/yellow_tripdata_2025-01.parquet")
df["duration"] = (df.tpep_dropoff_datetime - df.tpep_pickup_datetime).dt.total_seconds() / 60
df = df[(df.duration >= 1) & (df.duration <= 60)]
print(df.shape)
df[["PULocationID", "DOLocationID", "trip_distance", "duration"]].head()

(3403248, 21)


,PULocationID,DOLocationID,trip_distance,duration
0,229,237,1.60,8.350000
1,236,237,0.50,2.550000
2,141,141,0.60,1.950000
3,244,244,0.52,5.566667
4,244,116,0.66,3.533333


In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def load(path):
    df = pd.read_parquet(path)
    df["duration"] = (df.tpep_dropoff_datetime - df.tpep_pickup_datetime).dt.total_seconds() / 60
    return df[(df.duration >= 1) & (df.duration <= 60)]

train = load("../data/yellow_tripdata_2025-01.parquet")
val = load("../data/yellow_tripdata_2025-02.parquet")

features = ["PULocationID", "DOLocationID", "trip_distance"]
X_train, y_train = train[features], train["duration"]
X_val, y_val = val[features], val["duration"]

lr = LinearRegression().fit(X_train, y_train)
rmse = mean_squared_error(y_val, lr.predict(X_val)) ** 0.5
print("Linear regression RMSE:", rmse)

Linear regression RMSE: 9.619768190969548


In [3]:
from sklearn.ensemble import RandomForestRegressor

sample = train.sample(300_000, random_state=42)
rf = RandomForestRegressor(n_estimators=50, max_depth=15, n_jobs=-1, random_state=42)
rf.fit(sample[features], sample["duration"])
rmse_rf = mean_squared_error(y_val, rf.predict(X_val)) ** 0.5
print("Random forest RMSE:", rmse_rf)

Random forest RMSE: 4.925520439535689
